In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import chi2,SelectKBest
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
df=pd.read_csv("Titanic-Dataset.csv")

In [3]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
df.dtypes

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object

In [5]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [6]:
df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [10]:
df['Age']=df['Age'].fillna(df['Age'].median())
df['Embarked']=df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare']=df['Fare'].fillna(df['Fare'].median())

In [11]:
df['FamilySize']=df['SibSp']+df['Parch']

In [13]:
df['IsAlone']=np.where( df['FamilySize']==1,1,0)

In [14]:
df['IsAlone'] = np.where(df['FamilySize'] == 1, 1, 0)

In [15]:
df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=[0, 1, 2, 3, 4]
)


In [16]:
le=LabelEncoder()

In [17]:
for col in ['Sex','Embarked']:
    df[col]=le.fit_transform(df[col])

In [18]:
df['AgeGroup'] = df['AgeGroup'].astype(int)

In [20]:
features = [
    'Pclass',
    'Sex',
    'Age',
    'Fare',
    'Embarked',
    'FamilySize',
    'IsAlone',
    'AgeGroup'
]


In [21]:
X = df[features]
y = df['Survived']


In [23]:
selcetor=SelectKBest(score_func=chi2,k=5)

In [24]:
x_selected=selcetor.fit_transform(X,y)

In [26]:
selected_feature=X.columns[selcetor.get_support()]

In [27]:
print("\nTop Selected Features:")
print(selected_feature.tolist())



Top Selected Features:
['Pclass', 'Sex', 'Age', 'Fare', 'IsAlone']


In [37]:
scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selcetor.scores_
})


In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X[selected_feature],
    y,
    test_size=0.2,
    random_state=42
)
sorted_scores = scores.sort_values(by="Score", ascending=False)
print("\nFeature Scores:")
print(sorted_scores)


Feature Scores:
      Feature        Score
3        Fare  4518.319091
1         Sex    92.702447
0      Pclass    30.873699
2         Age    21.649163
6     IsAlone    19.432711
4    Embarked    10.202525
7    AgeGroup     2.679987
5  FamilySize     0.709091


In [40]:
model=RandomForestClassifier(n_estimators=100,random_state=42)

In [41]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", round(accuracy * 100, 2), "%")



Model Accuracy: 80.45 %


In [43]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Step 3: Load Dataset
# Select titanic.csv from your D drive when prompted

df = pd.read_csv("Titanic-Dataset.csv")

print("Dataset Loaded Successfully")
print("\nFirst 5 Records:")
print(df.head())

# =====================================================
# Step 4: Handle Missing Values
# =====================================================

df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

# =====================================================
# Step 5: Feature Engineering
# =====================================================

# Family Size
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Is Alone
df['IsAlone'] = np.where(df['FamilySize'] == 1, 1, 0)

# Age Group
df['AgeGroup'] = pd.cut(
    df['Age'],
    bins=[0, 12, 18, 35, 60, 100],
    labels=[0, 1, 2, 3, 4]
)

# Extract Title from Name
df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

rare_titles = ['Lady', 'Countess', 'Capt', 'Col', 'Don',
               'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']

df['Title'] = df['Title'].replace(rare_titles, 'Rare')
df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
df['Title'] = df['Title'].replace('Mme', 'Mrs')

# =====================================================
# Step 6: Encode Categorical Variables
# =====================================================

le = LabelEncoder()

for col in ['Sex', 'Embarked', 'Title']:
    df[col] = le.fit_transform(df[col])

df['AgeGroup'] = df['AgeGroup'].astype(int)

# =====================================================
# Step 7: Select Features
# =====================================================

features = [
    'Pclass',
    'Sex',
    'Age',
    'Fare',
    'Embarked',
    'FamilySize',
    'IsAlone',
    'AgeGroup',
    'Title'
]

X = df[features]
y = df['Survived']

# =====================================================
# Step 8: Feature Selection using Chi-Square
# =====================================================

selector = SelectKBest(score_func=chi2, k=5)

X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]

print("\nTop Selected Features:")
print(selected_features.tolist())

scores = pd.DataFrame({
    'Feature': X.columns,
    'Score': selector.scores_
})

print("\nFeature Scores:")
print(scores.sort_values(by='Score', ascending=False))

# =====================================================
# Step 9: Train Prediction Model
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X[selected_features],
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("\nModel Accuracy:", round(accuracy * 100, 2), "%")


Dataset Loaded Successfully

First 5 Records:
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0 

<>:45: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
<>:45: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
/var/folders/jv/3jcqz5md76gc2vx6rbbd25wr0000gn/T/ipykernel_24504/1694567364.py:45: SyntaxWarning: "\." is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\."? A raw string is also an option.
  df['Title'] = df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)
